# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/home/btaing14/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score, accuracy_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-20 13:42:27.709942: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-20 13:42:27.714940: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 13:42:27.774120: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-20 13:42:27.774186: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-20 13:42:27.774256: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#
X = df.drop(columns=["Class"]).values
y = df["Class"].values

In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 1s 23ms/step - loss: 1.1301 - accuracy: 0.3434 - val_loss: 0.9125 - val_accuracy: 0.5200
Epoch 2/20
13/13 [==============================] - 0s 7ms/step - loss: 0.8145 - accuracy: 0.7273 - val_loss: 0.6771 - val_accuracy: 0.8400
Epoch 3/20
13/13 [==============================] - 0s 6ms/step - loss: 0.5747 - accuracy: 0.9596 - val_loss: 0.4831 - val_accuracy: 1.0000
Epoch 4/20
13/13 [==============================] - 0s 6ms/step - loss: 0.3821 - accuracy: 0.9899 - val_loss: 0.3264 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 6ms/step - loss: 0.2448 - accuracy: 0.9899 - val_loss: 0.2224 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 6ms/step - loss: 0.1612 - accuracy: 0.9899 - val_loss: 0.1634 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 6ms/step - loss: 0.1123 - accuracy: 0.9899 - val_loss: 0.1251 - val_accuracy: 0.9600
Epoch 8/20
13/13 [=

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
# Predict class probabilities on the test set
y_pred_probs = model.predict(X_test)

# Convert probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Evaluate test accuracy
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"Test Accuracy: {test_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

2/2 [==============================] - 0s 6ms/step
Test Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
# Convert the trained Keras model to TFLite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the TFLite model
with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)

# Print file size in KB
model_base_size_kb = len(tflite_model) / 1024
print(f"Base TFLite model size: {model_base_size_kb:.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmppej88e5t/assets


INFO:tensorflow:Assets written to: /tmp/tmppej88e5t/assets


Base TFLite model size: 14.06 KB


2026-05-20 13:42:35.503927: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:42:35.504012: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:42:35.504421: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmppej88e5t
2026-05-20 13:42:35.506185: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:42:35.506208: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmppej88e5t
2026-05-20 13:42:35.512684: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 13:42:35.513595: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:42:35.558674: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmppej88e5t
2026-05

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def file_size_kb(filename):
    with open(filename, "rb") as f:
        return len(f.read()) / 1024
        
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # <-- Enter your code here <--#
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    # <-- Enter your code here <--#
    tflite_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # <-- Enter your code here for TFLite inference <--#
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_index = input_details["index"]
    output_index = output_details["index"]

    input_dtype = input_details["dtype"]
    output_dtype = output_details["dtype"]

    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    y_pred = []

    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)

        if input_dtype == np.int8 or input_dtype == np.uint8:
            input_data = input_data / input_scale + input_zero_point
            input_data= np.round(input_data).astype(input_dtype)
        else:
            input_data = input_data.astype(input_dtype)

        interpreter.set_tensor(input_index, input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_index)

        if output_dtype == np.int8 or output_dtype == np.uint8:
            output_data = (output_data.astype(np.float32) - output_zero_point) * output_scale

        pred_class = np.argmax(output_data, axis=1)[0]
        y_pred.append(pred_class)

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print(f"{quant_type.upper()} Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return y_pred

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# <-- Enter your code here <--#
y_pred_int8 = quantize_and_evaluate(
    model,
    X_test,
    y_test_cat,
    quant_type='int8',
    filename='model_int8.tflite'
)

y_pred_float16 = quantize_and_evaluate(
    model,
    X_test,
    y_test_cat,
    quant_type='float16',
    filename='model_float16.tflite'
)

y_pred_dynamic = quantize_and_evaluate(
    model,
    X_test,
    y_test_cat,
    quant_type='dynamic',
    filename='model_dynamic.tflite'
)

INFO:tensorflow:Assets written to: /tmp/tmpry_uhb1h/assets


INFO:tensorflow:Assets written to: /tmp/tmpry_uhb1h/assets
/home/btaing14/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 13:42:36.473078: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:42:36.473150: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:42:36.473397: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpry_uhb1h
2026-05-20 13:42:36.474729: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:42:36.474753: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpry_uhb1h
2026-05-20 13:42:36.477339: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.



INT8 TFLite model size: 5.73 KB
INT8 Test Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmpqmk7cw39/assets


INFO:tensorflow:Assets written to: /tmp/tmpqmk7cw39/assets
2026-05-20 13:42:37.197672: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:42:37.197748: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:42:37.198006: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpqmk7cw39
2026-05-20 13:42:37.198825: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:42:37.198841: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpqmk7cw39
2026-05-20 13:42:37.201132: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:42:37.242307: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpqmk7cw39
2026-05-20 13:42:37.253532: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


FLOAT16 TFLite model size: 8.94 KB
FLOAT16 Test Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /tmp/tmpslzqhi3g/assets


INFO:tensorflow:Assets written to: /tmp/tmpslzqhi3g/assets
2026-05-20 13:42:37.871288: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:42:37.871364: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:42:37.871584: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpslzqhi3g
2026-05-20 13:42:37.872447: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:42:37.872475: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpslzqhi3g
2026-05-20 13:42:37.874515: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:42:37.914368: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpslzqhi3g
2026-05-20 13:42:37.925381: I tensorflow/cc/saved_model/loader.cc:316] SavedModel


DYNAMIC TFLite model size: 8.16 KB
DYNAMIC Test Accuracy: 0.9815

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54


Confusion Matrix:
[[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# <-- Enter your code here <--#
batch_size = 8
epochs = 20

end_step = np.ceil(len(X_train) / batch_size).astype(np.int32) * epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# <-- Enter your code here <--#
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),

    prune_low_magnitude(
        tf.keras.layers.Dense(64, activation='relu'),
        pruning_schedule=pruning_schedule
    ),

    prune_low_magnitude(
        tf.keras.layers.Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),

    prune_low_magnitude(
        tf.keras.layers.Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# <-- Enter your code here <--#
pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

pruning_callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

pruned_history = pruned_model.fit(
    X_train,
    y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=pruning_callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 2s 19ms/step - loss: 1.0259 - accuracy: 0.4444 - val_loss: 0.9136 - val_accuracy: 0.7600
Epoch 2/10
13/13 [==============================] - 0s 6ms/step - loss: 0.7847 - accuracy: 0.7677 - val_loss: 0.7024 - val_accuracy: 0.8400
Epoch 3/10
13/13 [==============================] - 0s 8ms/step - loss: 0.6111 - accuracy: 0.9192 - val_loss: 0.5477 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 8ms/step - loss: 0.4671 - accuracy: 0.9697 - val_loss: 0.4157 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 8ms/step - loss: 0.3499 - accuracy: 0.9697 - val_loss: 0.3141 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 7ms/step - loss: 0.2526 - accuracy: 0.9798 - val_loss: 0.2377 - val_accuracy: 0.9200
Epoch 7/10
13/13 [==============================] - 0s 6ms/step - loss: 0.1792 - accuracy: 0.9899 - val_loss: 0.1821 - val_accuracy: 0.9200
Epoch 8/10
13/13 [=

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# <-- Enter your code here <--#
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
converter.optimizations = [tf.lite.Optimize.EXPERIMENTAL_SPARSITY]

pruned_tflite_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(pruned_tflite_model)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmp3pki2oqh/assets


INFO:tensorflow:Assets written to: /tmp/tmp3pki2oqh/assets
2026-05-20 13:42:41.899173: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:42:41.899249: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.


Pruned TFLite model size: 8.98 KB


2026-05-20 13:42:41.899451: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp3pki2oqh
2026-05-20 13:42:41.899912: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:42:41.899925: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp3pki2oqh
2026-05-20 13:42:41.901171: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:42:41.917394: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp3pki2oqh
2026-05-20 13:42:41.923554: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 24104 microseconds.


In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_pruned_probs = stripped_pruned_model.predict(X_test)
y_pred_pruned = np.argmax(y_pred_pruned_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

pruned_acc = accuracy_score(y_true, y_pred_pruned)

print(f"Pruned Test Accuracy: {pruned_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred_pruned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_pruned))

2/2 [==============================] - 0s 4ms/step
Pruned Test Accuracy: 0.9444

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.86      0.92        21
           2       0.88      1.00      0.94        15

    accuracy                           0.94        54
   macro avg       0.94      0.95      0.94        54
weighted avg       0.95      0.94      0.94        54


Confusion Matrix:
[[18  0  0]
 [ 1 18  2]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# <-- Enter your code here <--#
student_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# <-- Enter your code here <--#
teacher_soft_labels = model.predict(X_train)

4/4 [==============================] - 0s 3ms/step


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# <-- Enter your code here <--#
teacher_preds_soft = teacher_soft_labels

y_train_combined = np.concatenate(
    [y_train_cat, teacher_preds_soft],
    axis=1
)

def distillation_loss(y_true_combined, y_pred):

    # <-- Enter your code here: implement hard/soft label separation and weighted loss <--#
    alpha = 0.5

    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss

In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# <-- Enter your code here <--#
student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

student_history = student_model.fit(
    X_train,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 18ms/step - loss: 1.0292 - accuracy: 0.4949 - val_loss: 0.8350 - val_accuracy: 0.7600
Epoch 2/10
13/13 [==============================] - 0s 5ms/step - loss: 0.8142 - accuracy: 0.7778 - val_loss: 0.6799 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 5ms/step - loss: 0.6489 - accuracy: 0.8788 - val_loss: 0.5603 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 5ms/step - loss: 0.5231 - accuracy: 0.9394 - val_loss: 0.4674 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 5ms/step - loss: 0.4228 - accuracy: 0.9495 - val_loss: 0.3905 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 5ms/step - loss: 0.3443 - accuracy: 0.9596 - val_loss: 0.3296 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 5ms/step - loss: 0.2837 - accuracy: 0.9798 - val_loss: 0.2818 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# <-- Enter your code here <--#
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
kd_tflite_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(kd_tflite_model)

print(f"Knowledge Distillation TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpfoaixwls/assets


INFO:tensorflow:Assets written to: /tmp/tmpfoaixwls/assets


Knowledge Distillation TFLite model size: 6.09 KB


2026-05-20 13:42:44.783420: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:42:44.783520: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:42:44.783747: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpfoaixwls
2026-05-20 13:42:44.784461: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:42:44.784478: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpfoaixwls
2026-05-20 13:42:44.786523: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:42:44.826890: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpfoaixwls
2026-05-20 13:42:44.838826: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 55082 m

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# <-- Enter your code here <--#
y_pred_kd_probs = student_model.predict(X_test)
y_pred_kd = np.argmax(y_pred_kd_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

kd_acc = accuracy_score(y_true, y_pred_kd)

print(f"Knowledge Distillation Test Accuracy: {kd_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred_kd))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_kd))

2/2 [==============================] - 0s 5ms/step
Knowledge Distillation Test Accuracy: 0.9444

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.90      0.93        21
           2       0.93      0.93      0.93        15

    accuracy                           0.94        54
   macro avg       0.94      0.95      0.94        54
weighted avg       0.94      0.94      0.94        54


Confusion Matrix:
[[18  0  0]
 [ 1 19  1]
 [ 0  1 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# <-- (if needed) Enter your code here <--#
# Approach: Apply full INT8 quantization to the knowledge-distilled student model

y_pred_kd_int8 = quantize_and_evaluate(
    student_model,
    X_test,
    y_test_cat,
    quant_type='int8',
    filename='model_kd_int8.tflite'
)

INFO:tensorflow:Assets written to: /tmp/tmp4n5hcwh9/assets


INFO:tensorflow:Assets written to: /tmp/tmp4n5hcwh9/assets
/home/btaing14/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 3.61 KB
INT8 Test Accuracy: 0.9444

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.95      0.90      0.93        21
           2       0.93      0.93      0.93        15

    accuracy                           0.94        54
   macro avg       0.94      0.95      0.94        54
weighted avg       0.94      0.94      0.94        54


Confusion Matrix:
[[18  0  0]
 [ 1 19  1]
 [ 0  1 14]]


2026-05-20 13:42:45.622026: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 13:42:45.622096: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 13:42:45.622340: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp4n5hcwh9
2026-05-20 13:42:45.623580: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 13:42:45.623596: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp4n5hcwh9
2026-05-20 13:42:45.627158: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 13:42:45.667543: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp4n5hcwh9
2026-05-20 13:42:45.679292: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 56953 m

In comparison, the smallest model was the INT8 baseline at 5.73 KB, while the best-performing model was the KD student model at 0.9815 accuracy. Using my approach, it was 3.61 KB and achieved a test accuracy of 0.9444. This is smaller than the previous smallest model, and still maintains strong classification performance. Therefore, further size reduction was possible without a major accuracy loss.

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
